# 07 · k-NN recall — replacing the broken coverage metric

Closes **checklist item 9**.

## What was wrong

The coverage figures in the earlier draft — 0/29, 4/29, 23/29, 29/29 at four radii — are a
**radius sweep in an unnormalised 512-dimensional embedding space**. Three problems with
that as a coverage measure:

1. The radii are arbitrary and carry no units. In 512 dimensions almost all pairwise
   distances concentrate in a narrow band, so a sweep across four hand-picked radii mostly
   reports where that band happens to sit, not how well the synthetic set covers the real one.
2. Going 0/29 → 29/29 over a 4× range in radius is the signature of that concentration, not
   of a coverage threshold being crossed.
3. It never asks the question coverage is *for*: given a real failure, is there a synthetic
   case that behaves like it?

## What replaces it

**k-NN recall.** For each real failure, look at its k nearest neighbours among the synthetic
images and ask whether any of them is also a failure. Rank-based, so it needs no radius and
is invariant to the distance scale. Reported against two null models, because a recall
number alone is uninterpretable:

- **label-shuffled** — synthetic failure labels permuted. Gives the value chance produces at
  this base rate.
- **random-neighbour** — k synthetic images drawn uniformly instead of by distance. Isolates
  how much the *embedding geometry* contributes, as opposed to the base rate.

Cosine distance on L2-normalised embeddings, since BiomedCLIP is trained with a cosine
objective and Euclidean distance on unnormalised vectors mixes in magnitude.

## Inputs

Real failure labels now exist — they come from `02_results/02_baselines/per_nodule*.csv`,
produced by notebook 03. **Prefer the native pass** (`per_nodule_native.csv`) since
that is the in-distribution measurement; the notebook says which one it used.

CPU only, under a minute.

## 1 · Paths and inputs

In [ ]:
# ===========================================================================
# CANONICAL DRIVE PATHS
# Mirrored from src/paths.py and notebooks/CONFIG_CELL.md. Mapped from Drive
# 2026-09-12 with the Drive connector. Change all three together.
# Full layout, folder ids and the old->new table: DRIVE_LAYOUT.md
# ===========================================================================
!pip -q install -q scikit-learn
import os, json, glob, time, shutil, subprocess, random
from pathlib import Path
from google.colab import drive

# --- mount -----------------------------------------------------------------
# ismount(), not isdir(). A plain local directory under an unmounted
# /content/drive is also a dir, and creating one blocks the mount and makes
# Drive look empty -- that happened once and looked like a wiped Drive.
if not os.path.ismount('/content/drive'):
    if Path('/content/drive').exists():
        os.system('fusermount -u /content/drive 2>/dev/null')
        shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'mount failed'

# --- resolve the root ------------------------------------------------------
# MyDrive/Algoverse is a SHORTCUT to the shared folder Feliciano_Algoverse.
# One folder, not two; the FUSE mount resolves it as a directory.
# Test for the 01_data marker, never for the root itself: an unresolved
# shortcut and a stale empty directory both "exist", and that is exactly how
# a stray empty results/ tree got created on 2026-09-12.
ROOT = None
for _c in ['/content/drive/MyDrive/Algoverse',
           '/content/drive/MyDrive/Feliciano_Algoverse',
           '/content/drive/Shareddrives/Feliciano_Algoverse']:
    if (Path(_c)/'01_data').is_dir():
        ROOT = Path(_c); break
assert ROOT is not None, (
    'Algoverse root not found. Tried MyDrive/Algoverse, '
    'MyDrive/Feliciano_Algoverse, Shareddrives/Feliciano_Algoverse.\n'
    f'MyDrive top level: '
    f'{sorted(p.name for p in Path("/content/drive/MyDrive").iterdir())[:20]}\n'
    'MyDrive/Algoverse is a shortcut to the shared Feliciano_Algoverse. If it '
    'is gone: Drive -> Shared with me -> right-click Feliciano_Algoverse -> '
    'Add shortcut to Drive -> My Drive.')

# --- layout ----------------------------------------------------------------
SOURCE    = ROOT/'01_data'/'00_source'           # node21, chexpert, mimic_cxr
NODE21    = SOURCE/'node21'
MHA_SRC   = NODE21/'images'                      # 4,882 .mha
ANN_CSV   = NODE21/'metadata.csv'                # 5,224 rows, 1,476 label==1
GRID      = ROOT/'01_data'/'01_grid'
GRID_CSV  = GRID/'grid_v5.csv'                   # 231,145 bytes if it is the right one
RUNS      = GRID/'_runs'                         # generation checkpoint zips, not data
EMB_DIR   = ROOT/'01_data'/'02_embeddings'
CPASTE    = ROOT/'01_data'/'03_copypaste'
MODELS    = ROOT/'02_results'/'00_models'
CKPT      = MODELS/'baseline1_checkpoint.pth'    # .pth -- the old .pt path is dead
FIGS      = ROOT/'02_results'/'01_figures'
B1_DIR    = ROOT/'02_results'/'02_baselines'
PRED_DIR  = ROOT/'02_results'/'03_predictor'
B3_DIR    = ROOT/'02_results'/'04_baseline3'

print(f'root: {ROOT}')

OUT = Path('/content/knn'); OUT.mkdir(parents=True, exist_ok=True)
DEST = PRED_DIR
DEST.mkdir(parents=True, exist_ok=True)
for _p in (GRID_CSV, EMB_DIR/'synth_full.npz', EMB_DIR/'real_full.npz'):
    assert _p.exists(), f'{_p} missing -- see DRIVE_LAYOUT.md'
print(f'inputs ok\nwriting to {DEST}')

In [ ]:
import numpy as np, pandas as pd

def load_emb(path):
    z = np.load(path, allow_pickle=False)
    keys = list(z.keys())
    if len(keys) == 1 and z[keys[0]].ndim == 2:
        raise RuntimeError(f'{path.name} is a bare matrix with no ids')
    return {k: z[k].astype(np.float32).ravel() for k in keys}

E_syn  = load_emb(EMB_DIR/'synth_full.npz')
E_real = load_emb(EMB_DIR/'real_full.npz')
print(f'synth {len(E_syn)}  real {len(E_real)}  dim {len(next(iter(E_syn.values())))}')

g = pd.read_csv(GRID_CSV, keep_default_na=False, na_values=[''])

# ---- real failure labels, from notebook 03 --------------------------------
SRC = None
for cand in ['per_nodule_native.csv', 'per_nodule.csv']:
    if (B1_DIR/cand).exists():
        SRC = B1_DIR/cand; break
assert SRC is not None, (
    f'no per_nodule csv in {B1_DIR}. Run notebook 03 first -- section 9 for the native '
    'pass, which is the one to prefer.')
NATIVE = SRC.name.endswith('_native.csv')
print(f'\nreal labels from {SRC.name}'
      + ('   (native pass, in-distribution)' if NATIVE
         else '   (512+CLAHE pass -- OUT OF DISTRIBUTION, see FINDINGS 4b;'
              ' rerun 03 section 9 and prefer the native file)'))
nod = pd.read_csv(SRC)
print(f'{len(nod)} nodules over {nod.img_name.nunique()} images')

## 2 · Build the two label sets

A real **image** counts as a failure if any of its nodules is missed at the operating point
— the case-level definition, consistent with FINDINGS §6, because the embeddings are
whole-image and there is one vector per image, not per nodule.

Synthetic failures come from the grid restricted to the lesion arm with a clean background,
matching how failure is defined throughout.

In [ ]:
OP = 0.05

real_fail = (nod.assign(miss=nod.best_score < OP)
                .groupby('img_name').miss.any())
real_ids = [i for i in real_fail.index if i in E_real]
print(f'{len(real_ids)} of {len(real_fail)} real images have an embedding')
yr = real_fail.loc[real_ids].values.astype(bool)
print(f'real failures: {yr.sum()}/{len(yr)} ({yr.mean():.4f})')

syn = g[(g.arm == 'lesion') & (g.det_background == 0)].copy()
syn = syn[syn.image_id.isin(E_syn)]
ys = (syn.det_edited < OP).values.astype(bool)
syn_ids = syn.image_id.tolist()
print(f'synthetic: {len(syn_ids)} images, failures {ys.sum()} ({ys.mean():.4f})')
assert yr.sum() >= 5 and ys.sum() >= 5, 'too few failures on one side to be meaningful'

def l2(M):
    return M/np.clip(np.linalg.norm(M, axis=1, keepdims=True), 1e-8, None)

R = l2(np.stack([E_real[i] for i in real_ids]))
S = l2(np.stack([E_syn[i]  for i in syn_ids]))
print(f'\nR {R.shape}  S {S.shape}  (L2-normalised, cosine distance)')

## 3 · k-NN recall against both nulls

`recall@k` = of the real failures, the fraction with at least one synthetic failure among
their k nearest synthetic neighbours.

Both nulls are averaged over 200 draws so the comparison is not one lucky permutation.

In [ ]:
SIMS = R @ S.T                       # cosine similarity, real x synth
ORDER = np.argsort(-SIMS, axis=1)    # nearest first
rng = np.random.default_rng(0)
KS = [1, 3, 5, 10, 20, 50]
NDRAW = 200

def recall_at(k, labels, order):
    hit = labels[order[:, :k]].any(axis=1)
    return hit[yr].mean()

rows = []
for k in KS:
    obs = recall_at(k, ys, ORDER)
    sh = [recall_at(k, rng.permutation(ys), ORDER) for _ in range(NDRAW)]
    rd = []
    for _ in range(NDRAW):
        ro = rng.integers(0, len(syn_ids), size=(len(real_ids), k))
        rd.append(ys[ro].any(axis=1)[yr].mean())
    rows.append(dict(k=k, recall=round(obs, 4),
                     shuffled_mean=round(np.mean(sh), 4),
                     shuffled_p95=round(np.percentile(sh, 95), 4),
                     random_nbr_mean=round(np.mean(rd), 4),
                     lift_over_shuffled=round(obs-np.mean(sh), 4),
                     exceeds_p95=bool(obs > np.percentile(sh, 95))))
    print(f'  k={k:3}  recall {obs:.4f}   shuffled {np.mean(sh):.4f} '
          f'(p95 {np.percentile(sh,95):.4f})   random-nbr {np.mean(rd):.4f}   '
          f'{"ABOVE p95" if obs > np.percentile(sh,95) else "within null"}')

KNN = pd.DataFrame(rows)
KNN.insert(0, 'label_source', SRC.name)
KNN.to_csv(OUT/'table-knn-coverage.csv', index=False)
import shutil; shutil.copy(OUT/'table-knn-coverage.csv', DEST)
print()
print(KNN.to_string(index=False))

## 4 · Reverse direction, and the verdict

In [ ]:
# Reverse: for each synthetic failure, is a real failure among its k nearest real images?
# Coverage is directional and reporting only one side overstates it.
SIMS_T = S @ R.T
ORDER_T = np.argsort(-SIMS_T, axis=1)
rev = []
for k in KS:
    obs = yr[ORDER_T[:, :k]].any(axis=1)[ys].mean()
    sh = [yr[rng.permutation(np.arange(len(yr)))][ORDER_T[:, :k]].any(axis=1)[ys].mean()
          for _ in range(50)]
    rev.append(dict(k=k, recall_synth_to_real=round(obs, 4),
                    shuffled_mean=round(np.mean(sh), 4)))
    print(f'  k={k:3}  synth->real recall {obs:.4f}   shuffled {np.mean(sh):.4f}')
REV = pd.DataFrame(rev)
REV.to_csv(OUT/'table-knn-coverage-reverse.csv', index=False)
shutil.copy(OUT/'table-knn-coverage-reverse.csv', DEST)

print('\n' + '='*74)
print('CHECKLIST ITEM 9 — VERDICT')
print('='*74)
best = KNN.sort_values('lift_over_shuffled', ascending=False).iloc[0]
print(f"\nlargest lift over the shuffled null: k={int(best.k)}, "
      f"recall {best.recall:.4f} vs {best.shuffled_mean:.4f} "
      f"(lift {best.lift_over_shuffled:+.4f})")
print(f"exceeds the shuffled 95th percentile at any k: "
      f"{bool(KNN.exceeds_p95.any())}")
print('\nhow to read it:')
print('  recall above the shuffled p95, and above random-neighbour')
print('     -> the synthetic set covers real failures in a way the embedding geometry')
print('        supports. Report recall@k with both nulls, and the reverse direction.')
print('  recall inside the null')
print('     -> the synthetic set does NOT cover real failures, which is a finding and is')
print('        consistent with FINDINGS section 1: the grid varies requested attributes')
print('        while real difficulty varies continuously in conspicuity.')
print('\ncaveats to carry into the paper:')
print('  - both sides are whole-image embeddings, so "coverage" is image-level, not lesion')
print('    level. A real image is a failure if ANY of its nodules is missed.')
if not NATIVE:
    print('  - real labels came from the 512+CLAHE pass, which FINDINGS 4b shows is out of')
    print('    distribution for this detector. Rerun notebook 03 section 9 and redo this.')
print('  - 12 source chests underlie every synthetic point; the synthetic side is far less')
print('    independent than its row count suggests.')
print(f'\ntables -> {DEST}')